In [2]:
import pandas as pd
import cartopy.crs as ccrs
import geopandas as gpd
import matplotlib.patheffects as pe
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

In [8]:
# # load variable datasets
# # shear
# u200 = xr.open_mfdataset(
#     "datasets/ERA5_1940/u200*.nc",
#     combine = "by_coords",
#     preprocess=lambda ds: ds.drop_vars("time_bnds", errors="ignore"),
# )
# u850 = xr.open_mfdataset(
#     "datasets/ERA5_1940/u850*.nc",
#     combine = "by_coords",
#     preprocess=lambda ds: ds.drop_vars("time_bnds", errors="ignore"),
# )
# v200 = xr.open_mfdataset(
#     "datasets/ERA5_1940/v200*.nc",
#     combine = "by_coords",
#     preprocess=lambda ds: ds.drop_vars("time_bnds", errors="ignore"),
# )
# v850 = xr.open_mfdataset(
#     "datasets/ERA5_1940/v850*.nc",
#     combine = "by_coords",
#     preprocess=lambda ds: ds.drop_vars("time_bnds", errors="ignore"),
# )
# rhum
rhum = xr.open_mfdataset(
    "datasets/ERA5_1940/rhum*.nc",
    combine="by_coords",
    preprocess=lambda ds: ds.drop_vars("time_bnds", errors="ignore"),
    parallel=True,
    data_vars="minimal",
    coords="minimal",
    compat="override",
)
rhum = rhum[["r"]]

# print(rhum)

# # rename columns
# u200 = u200.rename({
#     "valid_time": "time",
#     "latitude": "lat",
#     "longitude": "lon"
# })
# u850 = u850.rename({
#     "valid_time": "time",
#     "latitude": "lat",
#     "longitude": "lon"
# })
# v200 = v200.rename({
#     "valid_time": "time",
#     "latitude": "lat",
#     "longitude": "lon"
# })
# v850 = v850.rename({
#     "valid_time": "time",
#     "latitude": "lat",
#     "longitude": "lon"
# })
rhum = rhum.rename({
    "valid_time": "time",
    "latitude": "lat",
    "longitude": "lon"
})

print(rhum)

<xarray.Dataset> Size: 11GB
Dimensions:         (time: 13158, pressure_level: 1, lat: 361, lon: 581)
Coordinates:
  * time            (time) datetime64[ns] 105kB 1940-06-01 ... 2025-10-31
  * pressure_level  (pressure_level) float64 8B 600.0
  * lat             (lat) float64 3kB 90.0 89.75 89.5 89.25 ... 0.5 0.25 0.0
  * lon             (lon) float64 5kB -120.0 -119.8 -119.5 ... 24.5 24.75 25.0
    number          int64 8B ...
    expver          (time) object 105kB dask.array<chunksize=(4131,), meta=np.ndarray>
Data variables:
    r               (time, pressure_level, lat, lon) float32 11GB dask.array<chunksize=(1032, 1, 73, 117), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-10-21T23:30 GRIB to CDM+CF via cfgr

/var/folders/d8/dpf8hpcx6l34ky46wf5b_v2h0000gn/T/ipykernel_96496/1656455243.py:24: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'valid_time' ('valid_time',) The recommendation is to set join explicitly for this case.
  rhum = xr.open_mfdataset(


In [ ]:
# load land mask
land_ds = xr.open_dataset(
    "datasets/land_mask/land_mask_ERA5.nc"
)

# rename columns
land_ds = land_ds.rename({
    "valid_time": "time",
    "latitude": "lat",
    "longitude": "lon"
})

# sort both spatial coordinates
land_ds = land_ds.sortby("lon").sortby("lat")

# print(land_ds)

# extract land mask and remove dimensions
land = land_ds["lsm"].squeeze(drop=True)

# keep ocean cells where land fraction <= 0.5
ocean_mask = land <= 0.5

# # apply mask to the variable dataset
# u200_ocean = u200.where(ocean_mask)
# u850_ocean = u850.where(ocean_mask)
# v200_ocean = v200.where(ocean_mask)
# v850_ocean = v850.where(ocean_mask)
rhum_ocean = rhum.where(ocean_mask)

Variables: ['r']
Dimensions: Frozen({'time': 13158, 'pressure_level': 1, 'lat': 361, 'lon': 581})
Output chunks: Frozen({'time': (1032, 1032, 1032, 1032, 1032, 1032, 1032, 1032, 1032, 1032, 1032, 1032, 774), 'pressure_level': (1,), 'lat': (73, 48, 25, 73, 23, 50, 69), 'lon': (117, 77, 40, 117, 37, 80, 113)})
Output directory exists: True
Output path: /Users/allisonclancy/Research/SyCLoPS/research_clean/datasets/RHUM/post_processing/post_landmask/rhum600_1940-2025_landmasked_v2.nc


In [ ]:
# # save file
# u200_ocean.to_netcdf("datasets/u-wind/post_processing/post_landmask/uwnd200_1940-2025_landmasked.nc")
# u850_ocean.to_netcdf("datasets/u-wind/post_processing/post_landmask/uwnd850_1940-2025_landmasked.nc")
# v200_ocean.to_netcdf("datasets/v-wind/post_processing/post_landmask/vwnd200_1940-2025_landmasked.nc")
# v850_ocean.to_netcdf("datasets/v-wind/post_processing/post_landmask/vwnd850_1940-2025_landmasked.nc")
rhum_ocean.to_netcdf("datasets/RHUM/post_processing/post_landmask/rhum600_1940-2025_landmasked_v2.nc")


HDF5-DIAG: Error detected in HDF5 (1.14.6) thread 1:
  #000: H5F.c line 653 in H5Fcreate(): unable to synchronously create file
    major: File accessibility
    minor: Unable to create file
  #001: H5F.c line 608 in H5F__create_api_common(): unable to create file
    major: File accessibility
    minor: Unable to open file
  #002: H5VLcallback.c line 3445 in H5VL_file_create(): file create failed
    major: Virtual Object Layer
    minor: Unable to create file
  #003: H5VLcallback.c line 3411 in H5VL__file_create(): file create failed
    major: Virtual Object Layer
    minor: Unable to create file
  #004: H5VLnative_file.c line 94 in H5VL__native_file_create(): unable to create file
    major: File accessibility
    minor: Unable to open file
  #005: H5Fint.c line 1921 in H5F_open(): unable to truncate a file which is already open
    major: File accessibility
    minor: Unable to open file


PermissionError: [Errno 13] Permission denied: '/Users/allisonclancy/Research/SyCLoPS/research_clean/datasets/RHUM/post_processing/post_landmask/rhum600_1940-2025_landmasked_v2.nc'